In [46]:
import importlib
import recommender_model

importlib.reload(recommender_model)

<module 'recommender_model' from '/Users/hariniprabakaran/Downloads/movie-recommender-main/recommender_model.py'>

In [47]:
recommend_pearson = recommender_model.recommend_pearson
recommend_cosine = recommender_model.recommend_cosine

In [49]:
test_movies = [
    "Matrix, The (1999)",
    "Toy Story (1995)",
    "Forrest Gump (1994)",
    "Pulp Fiction (1994)",
    "Star Wars: Episode IV - A New Hope (1977)",
    "Jurassic Park (1993)",
    "Lion King, The (1994)",
    "Terminator 2: Judgment Day (1991)",
    "Shawshank Redemption, The (1994)",
    "Fugitive, The (1993)"
]

In [59]:
import time

def avg_runtime(func, movie, runs=100):
    start = time.perf_counter()
    for _ in range(runs): func(movie)
    return (time.perf_counter() - start) / runs

pearson_time = avg_runtime(recommend_pearson, "Matrix, The (1999)")
cosine_time = avg_runtime(recommend_cosine, "Matrix, The (1999)")

In [60]:
def diversity_score(recommender, movies):
    unique_movies = set()

    for movie in movies:
        recs = recommender(movie)

        if not isinstance(recs, str):
            unique_movies.update(recs.index)

    return len(unique_movies)

pearson_diversity = diversity_score(recommend_pearson, test_movies)
cosine_diversity = diversity_score(recommend_cosine, test_movies)

In [62]:
def reciprocal_recommendation_rate(recommender, movies):

    hits = total = 0

    for movie in movies:
        recs = recommender(movie)

        if isinstance(recs, str):
            continue

        for rec_movie in recs.index:
            reverse_recs = recommender(rec_movie)

            if not isinstance(reverse_recs, str):
                hits += movie in reverse_recs.index
                total += 1

    return hits / total if total else 0

pearson_rrr = reciprocal_recommendation_rate(recommend_pearson, test_movies)
cosine_rrr = reciprocal_recommendation_rate(recommend_cosine, test_movies)

In [63]:
results = pd.DataFrame({
    "Method": ["Pearson", "Cosine"],
    "Avg Runtime (s)": [pearson_time, cosine_time],
    "Diversity": [pearson_diversity, cosine_diversity],
    "Reciprocal Recommendation Rate": [
        pearson_rrr,
        cosine_rrr
    ]
})

results

,Method,Avg Runtime (s),Diversity,Reciprocal Recommendation Rate
0,Pearson,0.001766,63,0.54
1,Cosine,0.000697,49,0.85


#### Selection Criteria

| Metric | Winner |
|----------|----------|
| Runtime | Cosine Similarity |
| Diversity | Pearson Correlation |
| Reciprocal Recommendation Rate | Cosine Similarity |

#### Final Model Selection

**Cosine Similarity** was selected because it generated recommendations faster and achieved a higher Reciprocal Recommendation Rate, while still maintaining good recommendation diversity.